In [27]:
import pandas as pd
import numpy as np
import math, os


from utils import PARAMS

In [ ]:

# def analyze_fitts_trials(df, hold_required=120):

#     df = df.reset_index(drop=True)  # FIX

#     df["new_trial"] = (
#         (df["target_x"].shift() != df["target_x"]) |
#         (df["target_y"].shift() != df["target_y"])
#     )

#     trial_indices = df.index[df["new_trial"]].tolist()
#     if 0 not in trial_indices:
#         trial_indices.insert(0, 0)

#     results = []

#     for i in range(len(trial_indices)):
#         start_idx = trial_indices[i]
#         end_idx = trial_indices[i + 1] - 1 if i + 1 < len(trial_indices) else len(df) - 1

#         trial = df.iloc[start_idx:end_idx + 1]
#         if trial.empty:
#             continue

#         t_start = trial["time"].iloc[0]
#         t_end = trial["time"].iloc[-1]
#         MT = t_end - t_start

#         start_x, start_y = trial["cursor_x"].iloc[0], trial["cursor_y"].iloc[0]
#         target_x, target_y = trial["target_x"].iloc[0], trial["target_y"].iloc[0]
#         W = trial["radius"].iloc[0] * 2

#         D = math.hypot(target_x - start_x, target_y - start_y)
#         ID = math.log2(D / W + 1) if W > 0 else np.nan
#         TP = ID / MT if MT > 0 else np.nan

#         success = int(trial["hold_count"].max() >= hold_required - 1)

#         dx = np.diff(trial["cursor_x"].values)
#         dy = np.diff(trial["cursor_y"].values)
#         path_length = np.sum(np.sqrt(dx**2 + dy**2))
#         path_eff = D / path_length if path_length > 0 else np.nan

#         inside = trial["inside"].astype(bool).values
#         crossings = np.sum(inside[1:] != inside[:-1]) // 2

#         results.append({
#             "Trial": i + 1,
#             "MT": MT,
#             # "D": D,
#             # "W": W,
#             # "ID": ID,
#             "TP": TP,
#             "Win": success,
#             "PE": path_eff,
#             "Cross": crossings
#         })

#     results = pd.DataFrame(results)
#     return results, results.describe()

def analyze_fitts_trials(df, hold_required=120):

    df = df.reset_index(drop=True)

    df["new_trial"] = (
        (df["target_x"].shift() != df["target_x"]) |
        (df["target_y"].shift() != df["target_y"])
    )

    trial_indices = df.index[df["new_trial"]].tolist()
    if 0 not in trial_indices:
        trial_indices.insert(0, 0)

    results = []

    for i in range(len(trial_indices)):
        start_idx = trial_indices[i]
        end_idx = trial_indices[i + 1] - 1 if i + 1 < len(trial_indices) else len(df) - 1

        trial = df.iloc[start_idx:end_idx + 1]
        if trial.empty:
            continue

        t_start = trial["time"].iloc[0]

        inside = trial["inside"].astype(bool).values
        hold = trial["hold_count"].values

        # ===== ISO selection moment =====
        sel_idx = np.where(inside & (hold >= hold_required))[0]

        if sel_idx.size > 0:
            sel = sel_idx[0]
            t_end = trial["time"].iloc[sel]
            success = 1
        else:
            # fallback (timeout / failure)
            t_end = trial["time"].iloc[-1]
            success = 0

        MT = t_end - t_start

        start_x, start_y = trial["cursor_x"].iloc[0], trial["cursor_y"].iloc[0]
        target_x, target_y = trial["target_x"].iloc[0], trial["target_y"].iloc[0]
        W = trial["radius"].iloc[0] * 2

        D = math.hypot(target_x - start_x, target_y - start_y)
        ID = math.log2(D / W + 1) if W > 0 else np.nan
        TP = ID / MT if MT > 0 else np.nan

        dx = np.diff(trial["cursor_x"].values)
        dy = np.diff(trial["cursor_y"].values)
        path_length = np.sum(np.sqrt(dx**2 + dy**2))
        path_eff = D / path_length if path_length > 0 else np.nan

        crossings = np.sum(inside[1:] != inside[:-1]) // 2

        results.append({
            "Trial": i + 1,
            "MT": MT,
            "TP": TP,
            "Win": success,
            "PE": path_eff,
            "Cross": crossings
        })

    results = pd.DataFrame(results)
    return results, results.describe()

In [39]:
NAME = 'andrew_r_1'
path = f'fitts_logs/{NAME}/'

print(f'NAME: {NAME}')

files =  sorted(os.listdir(path))

for f in files:
    print('\n\n\n',f)
    df = pd.read_csv(path + f)
    trial_metrics, summary = analyze_fitts_trials(df, hold_required=PARAMS['hold_frames_required'])
    print(trial_metrics)
    print(summary)

NAME: andrew_r_1



 Fitts_cnn_raw_2026-02-20_15-18-30.csv
    Trial         MT  Win        PE  Cross           D     W          xe  \
0       1   2.828118    0  0.998668      0  300.000000  42.0   -1.000000   
1       2   0.000000    1  1.023235      1  599.001636  40.0 -599.001636   
2       3   0.000000    1  0.877743      1  575.097140  42.0 -575.097140   
3       4   0.000000    1  0.682395      1  587.951154  42.0 -587.951154   
4       5   0.000000    1  0.644711      1  575.597950  40.0 -575.597950   
5       6   0.000000    1  0.154847      0  604.585048  42.0 -604.585048   
6       7  14.383650    0  0.128996      0  327.649691  40.0    6.625125   
7       8  14.383928    0  0.190600      0  583.237482  40.0 -523.692337   
8       9   6.303717    0  0.188387      0  118.570823  40.0   -9.036962   
9      10   0.000000    1  0.162825      2  583.826858  40.0 -583.826858   
10     11   9.391061    0  0.453439      1  574.235666  40.0  -13.788207   
11     12   0.000000    1  0.